In [1]:
# 1. Pull the repository from GitHub
!git clone -q https://github.com/luissejas/OilPricePrediction.git

# 2. Move inside the project root
%cd OilPricePrediction

# 3. Install dependencies quietly
!pip install -q -r requirements.txt
!pip install -q torch transformers

# 4. Handle the Data Upload securely
import os
import shutil
from google.colab import files

os.makedirs('data', exist_ok=True)
print("Environment synchronized. Please select 'processed_oil_data.csv' to upload:")

uploaded = files.upload()

for filename in uploaded.keys():
    destination = f"data/{filename}"
    shutil.move(filename, destination)
    print(f"Success. Data secured in: {destination}")

/content/OilPricePrediction
Environment synchronized. Please select 'processed_oil_data.csv' to upload:


Saving processed_oil_data.csv to processed_oil_data.csv
Success. Data secured in: data/processed_oil_data.csv


In [2]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# 1. Hardware Gatekeeper
if not torch.cuda.is_available():
    print("CRITICAL ERROR: A GPU (CUDA) is required to run this script.")
    print("Please go to Runtime > Change runtime type > Hardware accelerator > T4 GPU.")
    sys.exit(1)

device = torch.device("cuda")
print(f"Hardware Check Passed. Executing on: {torch.cuda.get_device_name(0)}")

# 2. Seed Lock
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global random seed locked to {seed}")

set_seed(42)

# 3. Directory Prep
os.makedirs('src/models', exist_ok=True)

Hardware Check Passed. Executing on: Tesla T4
Global random seed locked to 42


In [3]:
%%writefile src/models/pytorch_model.py
import torch
import torch.nn as nn

class OilPriceNN(nn.Module):
    """
    Multi-Layer Perceptron for Oil Price Prediction.
    Architecture: Input -> 64 -> 32 -> Output (1)
    """
    def __init__(self, input_size):
        super(OilPriceNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

Writing src/models/pytorch_model.py


In [4]:
# Import the blueprint from the file we just generated
from src.models.pytorch_model import OilPriceNN

class PyTorchForecaster:
    def __init__(self, data_path="data/processed_oil_data.csv"):
        self.data_path = data_path
        self.scaler = StandardScaler()
        self.model = None

    def prepare_data(self):
        df = pd.read_csv(self.data_path, index_col='Date', parse_dates=True).dropna()
        feature_cols = ['price_lag_1', 'price_lag_3', 'sma_7', 'sma_14', 'volatility_7']

        X = df[feature_cols].values
        y = df['price'].values.reshape(-1, 1)

        split_idx = int(len(df) * 0.8)
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)

        return (torch.FloatTensor(X_train).to(device),
                torch.FloatTensor(X_test).to(device),
                torch.FloatTensor(y_train).to(device),
                torch.FloatTensor(y_test).to(device))

    def reset_model(self):
        if self.model is not None:
            del self.model
            self.model = None
        torch.cuda.empty_cache()

    def train(self, X_train, y_train, epochs=2000):
        self.reset_model()
        self.model = OilPriceNN(input_size=X_train.shape[1]).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.02)

        print(f"Training PyTorch Neural Network for {epochs} epochs...")
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = self.model(X_train)
            loss = criterion(outputs, y_train)
            loss.backward()
            optimizer.step()

            if (epoch+1) % 200 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    def evaluate(self, X_test, y_test):
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(X_test)
            y_test_cpu = y_test.cpu().numpy()
            predictions_cpu = predictions.cpu().numpy()

            mae = mean_absolute_error(y_test_cpu, predictions_cpu)
            print(f"\n[PyTorch MLP] Mean Absolute Error: ${mae:.2f} per barrel")
            return mae

    def save_if_better(self, current_mae, model_path="models/champion_nn.pth", scaler_path="models/nn_scaler.pkl", metric_path="config/nn_metrics.json"):
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        os.makedirs(os.path.dirname(metric_path), exist_ok=True)

        best_mae = float('inf')
        if os.path.exists(metric_path):
            with open(metric_path, 'r') as f:
                best_mae = json.load(f).get('best_mae', float('inf'))

        if current_mae < best_mae:
            print(f"NEW CHAMPION! MAE improved from ${best_mae:.2f} to ${current_mae:.2f}")
            torch.save(self.model.state_dict(), model_path)
            joblib.dump(self.scaler, scaler_path)

            with open(metric_path, 'w') as f:
                json.dump({'best_mae': current_mae}, f)

            print("Model, Scaler, and Config files successfully updated.")
        else:
            print(f"Challenger Defeated. Current MAE (${current_mae:.2f}) did not beat Champion (${best_mae:.2f}).")
            print("Old model files kept completely intact.")

In [5]:
if __name__ == "__main__":
    forecaster = PyTorchForecaster()
    X_train, X_test, y_train, y_test = forecaster.prepare_data()

    # Train the model
    forecaster.train(X_train, y_train, epochs=2000)

    # Evaluate and trigger the gatekeeper
    current_mae = forecaster.evaluate(X_test, y_test)
    forecaster.save_if_better(current_mae)

    # --- MANUAL SYNC REMINDER ---
    print("\n" + "="*65)
    print("ACTION REQUIRED: MANUAL LOCAL SYNC")
    print("="*65)
    print("Please download the following files and place them in your")
    print("local repository to preserve this model and your work:\n")
    print("From the Colab File Explorer (Left Menu):")
    print("  1. models/champion_nn.pth")
    print("  2. models/nn_scaler.pkl")
    print("  3. config/nn_metrics.json")
    print("  4. src/models/pytorch_model.py")
    print("From the Top Menu (File > Download > Download .ipynb):")
    print("  5. The current Colab Notebook (save to your experiments/ folder)")
    print("="*65 + "\n")

Training PyTorch Neural Network for 2000 epochs...
Epoch [200/2000], Loss: 6.2010
Epoch [400/2000], Loss: 5.1997
Epoch [600/2000], Loss: 5.0364
Epoch [800/2000], Loss: 4.9427
Epoch [1000/2000], Loss: 4.8585
Epoch [1200/2000], Loss: 4.7851
Epoch [1400/2000], Loss: 4.6981
Epoch [1600/2000], Loss: 4.6154
Epoch [1800/2000], Loss: 4.5241
Epoch [2000/2000], Loss: 4.4208

[PyTorch MLP] Mean Absolute Error: $1.09 per barrel
Challenger Defeated. Current MAE ($1.09) did not beat Champion ($1.09).
Old model files kept completely intact.

ACTION REQUIRED: MANUAL LOCAL SYNC
Please download the following files and place them in your
local repository to preserve this model and your work:

From the Colab File Explorer (Left Menu):
  1. models/champion_nn.pth
  2. models/nn_scaler.pkl
  3. config/nn_metrics.json
  4. src/models/pytorch_model.py  <-- The Architectural Blueprint

From the Top Menu (File > Download > Download .ipynb):
  5. The current Colab Notebook (save to your experiments/ folder)

